# 2. AG News Classification with an LSTM

**What this notebook teaches.** Text classification end to end: [AG News](http://groups.di.unipi.it/~gulli/AG_corpus_of_news_articles.html) headlines + descriptions, 4 classes (World, Sports, Business, Sci/Tech), 120,000 training and 7,600 test articles.

* cleaning the text and building a vocabulary of the 20,000 most frequent words (`<pad>` = 0, `<unk>` = 1);
* turning every article into a fixed-length sequence of word ids (padded at the front, so that the last time step is always a real word);
* a model with a **learned embedding layer** (trained from scratch, together with the rest), an LSTM that reads the sequence left to right, and a linear layer on the final hidden state;
* training with back-propagation through time (BPTT).

**New compared to notebook 1:** instead of using pretrained embeddings for inspection, the embeddings are learned for the task.

In [ ]:
import torch
import numpy as np
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm import tqdm

In [ ]:
# Hyperparameters
max_len = 64
vocab_size = 20_000
dim = 128
device = torch.device('cuda' if torch.cuda.is_available()
                      else 'mps' if torch.backends.mps.is_available() else 'cpu')
batch_size = 32
learning_rate = 1e-3
epochs = 5
seed = 2024

np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True

## Load data

In [ ]:
from utils import get_data
X_train, y_train, X_test, y_test = get_data()

In [ ]:
X_test[:10]

## Split data into words, calculate length of each sentence,
## and calculate the statistics of the dataset

In [ ]:
lens = []
for sentence in X_train:
    lens.append(len(sentence.split(' ')))
lens = np.array(lens)
lens.mean(), np.percentile(lens, 95), lens.max()

## Preprocess data and create the tensors

In [ ]:
# get the counts of all words within the first 64 tokens in a sentence
word_counts = {}
for sentence in X_train:
    for word in sentence.split(' ')[:max_len]:
        if word not in word_counts:
            word_counts[word] = 1
        else:
            word_counts[word] += 1
len(word_counts)

In [ ]:
sorted_words = sorted(word_counts.items(), key=lambda x: x[1], reverse=True)
sorted_words = sorted_words[:vocab_size]
sorted_words[-10:]

In [ ]:
# select top 20k words and create word2idx and idx2word
sorted_words = sorted(word_counts.items(), key=lambda x: x[1], reverse=True)
sorted_words = sorted_words[:vocab_size]
word2idx = {word: i+2 for i, (word, _) in enumerate(sorted_words)}
word2idx['<pad>'] = 0
word2idx['<unk>'] = 1
idx2word = {i: word for word, i in word2idx.items()}
len(word2idx)

In [ ]:
word2idx

In [ ]:
# prepare the data: words -> ids; unknown words -> <unk> (id 1)
def tokenize(sentences):
    out = []
    for sentence in sentences:
        sentence = sentence.split(' ')[:max_len]
        sentence = [word2idx.get(word, 1) for word in sentence]
        sentence = [0] * (max_len - len(sentence)) + sentence  # prepad the sentence
        out.append(sentence)
    return torch.tensor(out)

X_train_tensor = tokenize(X_train)
y_train_tensor = torch.tensor(y_train)
X_test_tensor = tokenize(X_test)
y_test_tensor = torch.tensor(y_test)

In [ ]:
# data loader
train_dataset = torch.utils.data.TensorDataset(X_train_tensor, y_train_tensor)
# train loader in train loop

test_dataset = torch.utils.data.TensorDataset(X_test_tensor, y_test_tensor)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

## Create Model

In [ ]:
class Model(nn.Module):
    def __init__(self, vocab_size, hid_dim, out_dim):
        super(Model, self).__init__()
        self.embedding = nn.Embedding(vocab_size + 2, hid_dim, padding_idx=0)
        self.rnn = nn.LSTM(hid_dim, hid_dim, batch_first=True)
        self.fc = nn.Linear(hid_dim, out_dim)
    
    def forward(self, x):
        x = self.embedding(x)
        x, _ = self.rnn(x)
        x = x[:, -1, :]
        x = self.fc(x)
        return x

In [ ]:
model = Model(vocab_size, dim, 4).to(device)
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

## Train

In [ ]:
for epoch in range(epochs):
    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True)
    model.train()
    for X_batch, y_batch in tqdm(train_loader):
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        logits = model(X_batch)
        log_probs = F.log_softmax(logits, dim=1)
        loss = F.nll_loss(log_probs, y_batch)
        
        optimizer.zero_grad()
        loss.backward()  # back-propagation through time (BPTT)
        optimizer.step()
    
    model.eval()
    with torch.no_grad():
        correct = 0
        total = 0
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            y_pred = F.log_softmax(y_pred, dim=1)
            _, predicted = torch.max(y_pred, 1)
            correct += (predicted == y_batch).sum().item()
            total += y_batch.size(0)
        acc = correct / total
    print(f'Epoch: {epoch+1:02d}, Test Acc: {acc:.4f}')